# HW 10: Bayesian Interception with a New Speed Distribution
**Computational Sensorimotor Control — Fall 2026**

A new experiment uses four target speeds: 40, 60, 80, 100°/s.
- **Block A:** all speeds equally likely (25% each)
- **Block B:** 80°/s appears on 70% of trials, the rest 10% each

Use the conjugate Bayesian framework from Lecture §7.

In [ ]:
import numpy as np

# Constants from lecture
MT_BIAS = 1.12
sigma_MT = 12.0       # MT velocity noise (°/s)
n_obs = 14            # MT observations from 60–200 ms
sigma_lik = sigma_MT / np.sqrt(n_obs)  # likelihood σ = 3.2°/s
T_cross = 0.550       # hand flight time (s)
T_launch = 0.245      # hand launch time (s)
HIT_WINDOW = 4.8      # ±4.8° = ±1.25 cm

speeds = [40, 60, 80, 100]  # °/s

# Block distributions
block_A = {40: 0.25, 60: 0.25, 80: 0.25, 100: 0.25}
block_B = {40: 0.10, 60: 0.10, 80: 0.70, 100: 0.10}

N_practice = 25  # practice trials for σ_prior

print('Setup complete.')

---
## Part 1: Conjugate Update

In [ ]:
# ══ Part 1: Compute priors and posterior velocity estimates ══

# Block A: flat prior
mu_A = ...  # ← YOUR CODE HERE: mean of the speed distribution
sigma_dist_A = np.sqrt(np.sum([p * (s - mu_A)**2 for s, p in block_A.items()]))
sigma_prior_A = ...  # ← YOUR CODE HERE: σ_distribution / √N
w_prior_A = (1/sigma_prior_A**2) / (1/sigma_prior_A**2 + 1/sigma_lik**2)

# Block B: biased prior — mode is 80°/s
mu_B = ...  # ← YOUR CODE HERE: statistical mode of Block B
sigma_dist_B = np.sqrt(np.sum([p * (s - mu_B)**2 for s, p in block_B.items()]))
sigma_prior_B = ...  # ← YOUR CODE HERE
w_prior_B = (1/sigma_prior_B**2) / (1/sigma_prior_B**2 + 1/sigma_lik**2)

print(f'Block A: μ_prior = {mu_A:.1f}°/s, σ_prior = {sigma_prior_A:.1f}°/s, w_prior = {w_prior_A:.3f}')
print(f'Block B: μ_prior = {mu_B:.1f}°/s, σ_prior = {sigma_prior_B:.1f}°/s, w_prior = {w_prior_B:.3f}')

# Posterior velocity estimates
print(f'\n{"Speed":>6s}  {"Block A E[ω̂]":>14s}  {"Block B E[ω̂]":>14s}  {"True":>6s}  {"MT biased":>10s}')
best_help = ''; worst_hurt = ''; best_diff = 0; worst_diff = 0
for omega in speeds:
    biased = omega * MT_BIAS
    post_A = w_prior_A * mu_A + (1 - w_prior_A) * biased
    post_B = w_prior_B * mu_B + (1 - w_prior_B) * biased
    err_A = abs(post_A - omega); err_B = abs(post_B - omega)
    diff = err_A - err_B  # positive = B helps
    if diff > best_diff: best_diff = diff; best_help = f'{omega}°/s'
    if diff < worst_diff: worst_diff = diff; worst_hurt = f'{omega}°/s'
    print(f'{omega:6d}  {post_A:14.1f}  {post_B:14.1f}  {omega:6d}  {biased:10.1f}')

print(f'\nBlock B helps most at {best_help} (error reduced by {best_diff:.1f}°)')
print(f'Block B hurts most at {worst_hurt} (error increased by {abs(worst_diff):.1f}°)')

---
## Part 2: Feedforward Miss

In [ ]:
# ══ Part 2: Feedforward miss for all speeds × both blocks ══
import matplotlib.pyplot as plt

miss_A = []; miss_B = []
for omega in speeds:
    biased = omega * MT_BIAS
    post_A = w_prior_A * mu_A + (1 - w_prior_A) * biased
    post_B = w_prior_B * mu_B + (1 - w_prior_B) * biased
    theta_200 = omega * 0.200
    aim_A = ...  # ← YOUR CODE HERE: θ(200ms) + posterior × T_cross
    aim_B = ...  # ← YOUR CODE HERE
    true_int = omega * (T_launch + T_cross)
    miss_A.append(aim_A - true_int)
    miss_B.append(aim_B - true_int)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(speeds))
ax.plot(x, miss_A, 'o-', color='#2E86AB', ms=8, lw=2, label='Block A (flat)')
ax.plot(x, miss_B, 's-', color='#E67E22', ms=8, lw=2, label='Block B (biased to 80)')
ax.axhspan(-HIT_WINDOW, HIT_WINDOW, alpha=0.08, color='#27AE60')
ax.axhline(HIT_WINDOW, color='#27AE60', ls='--', lw=1, alpha=0.4)
ax.axhline(-HIT_WINDOW, color='#27AE60', ls='--', lw=1, alpha=0.4)
ax.axhline(0, color='k', lw=0.5, alpha=0.3)
ax.set_xticks(x); ax.set_xticklabels([f'{s}°/s' for s in speeds])
ax.set_xlabel('Target speed (°/s)'); ax.set_ylabel('Feedforward miss (°)')
ax.set_title('HW10: Feedforward miss — Block A vs Block B', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Find first speed where Block B exceeds hit window
for i, omega in enumerate(speeds):
    if abs(miss_B[i]) > HIT_WINDOW:
        print(f'\nBlock B first exceeds hit window at {omega}°/s (miss = {miss_B[i]:.1f}°)')
        break
else:
    print('\nBlock B stays within hit window for all speeds')

print(f'\nAll misses:')
for i, omega in enumerate(speeds):
    print(f'  {omega}°/s: Block A = {miss_A[i]:+.1f}°  Block B = {miss_B[i]:+.1f}°')

---
## Part 3: Expected Miss

In [ ]:
# ══ Part 3: Expected absolute miss under each block's own distribution ══

E_miss_A = ...  # ← YOUR CODE HERE: Σ p_i × |miss_i|
E_miss_B = ...  # ← YOUR CODE HERE

print(f'Expected |miss|:')
print(f'  Block A (flat):    {E_miss_A:.2f}°')
print(f'  Block B (biased):  {E_miss_B:.2f}°')
print(f'  Difference:        {E_miss_B - E_miss_A:+.2f}°')

if E_miss_B > E_miss_A:
    print(f'\n→ Block B prior WORSENS overall performance by {E_miss_B - E_miss_A:.2f}°')
else:
    print(f'\n→ Block B prior IMPROVES overall performance by {E_miss_A - E_miss_B:.2f}°')

# One-sentence explanation (solution)
print('''
Explanation: The Block B prior pulls the velocity estimate toward 80°/s.
For the 70% of trials at 80°/s, this partially corrects the Aubert-Fleischl
overshoot (biased MT gives 89.6°/s, prior pulls it back toward 80). But for
the 10% of trials at 40°/s, the prior fights the likelihood in the wrong
direction — pushing the estimate UP from 44.8 toward 80, producing a large
overshoot. The Aubert-Fleischl bias and the speed prior compound rather than
cancel at speeds below the prior mode.
''')

---
**End of HW10.**